In [14]:
# requirements:
# %pip install requests beautifulsoup4 lxml tqdm By
%pip install rembg pillow numpy tqdm

  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached numpy-2.2.6-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl (38.9 MB)
   ---------------------------------------- 0.0/15.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/15.6 MB 8.4 MB/s eta 0:00:02
   ---------- ----------------------------- 4.2/15.6 MB 11.4 MB/s eta 0:00:01
   -------------- ------------------------- 5.8/15.6 MB 9.8 MB/s eta 0:00:02
   ------------------ --------------------- 7.1/15.6 MB 9.1 MB/s eta 0:00:01
   ---------------------- ----------------- 8.7/15.6 MB 8.7 MB/s eta 0:00:01
   -------------------------- ------------- 10.2/15.6 MB 8.5 MB/s eta 0:00:01
   ------------------------------ --------- 11.8/15.6 MB 8.3 MB/s eta 0:00:01
   ---------------------------------- ----- 13.4/15.6 MB 8.1 MB/s eta 0:00:01
   -------------------------------------- - 14.9/15.6 MB 8.0 MB/s eta 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.0.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.


In [ ]:
# requirements:
#   pip install selenium webdriver-manager requests tqdm

import os, re, time, random
from pathlib import Path
from urllib.parse import urljoin, urlparse
import requests
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# from webdriver_manager.chrome import ChromeDriverManager  # (Selenium Manager 사용 시 불필요)

BASE_URL  = "https://www.hyundai.com"    # 최종 이미지에 붙일 도메인
START_URL = "https://www.hyundai.com/kr/ko/e/menu-list/"   # #model 이 있는 실제 페이지 URL
OUT_DIR   = Path("insight&trends_images")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Referer": BASE_URL
}

# ---- Rate limit-friendly delays ----
REQ_DELAY_RANGE = (3.5, 7.0)   # 페이지 이동 사이 대기(랜덤)
IMG_DELAY_RANGE = (1.0, 2.2)   # 이미지 다운로드 사이 대기(랜덤)
BLOCK_PATTERNS = ["Error 1015", "You are being rate limited", "cf-error-code", "Ray ID"]

def polite_sleep(a, b):
    time.sleep(random.uniform(a, b))

def detect_block_html(html: str) -> bool:
    low = html.lower()
    return any(p.lower() in low for p in BLOCK_PATTERNS)

def polite_get(driver, url, wait, locator=None):
    polite_sleep(*REQ_DELAY_RANGE)
    driver.get(url)
    if locator:
        try:
            wait.until(EC.presence_of_element_located(locator))
        except:
            pass
    if detect_block_html(driver.page_source):
        raise RuntimeError(f"[BLOCK] Cloudflare rate limit 감지: {url}")

# -------- Utils --------
def extract_bg_url(style_text: str) -> str | None:
    """
    style="background-image: url('/contents/.../xxx.jpg');" 에서 /contents/... 추출
    url("..."), url('...'), url(...) 모두 대응
    """
    if not style_text:
        return None
    style_text = style_text.replace("&quot;", '"')
    m = re.search(r'background-image\s*:\s*url\((["\']?)([^)"\']+)\1\)', style_text, flags=re.I)
    if not m:
        m = re.search(r'url\((["\']?)([^)"\']+)\1\)', style_text, flags=re.I)
    return m.group(2) if m else None

def safe_filename(url: str) -> str:
    name = os.path.basename(urlparse(url).path) or "image"
    return re.sub(r'[^A-Za-z0-9._-]+', "_", name)

def download(url: str, referer: str | None = None):
    polite_sleep(*IMG_DELAY_RANGE)
    headers = dict(HEADERS)
    if referer:
        headers["Referer"] = referer
    with requests.get(url, headers=headers, stream=True, timeout=30) as r:
        r.raise_for_status()
        fname = safe_filename(url)
        dst = OUT_DIR / fname
        # 중복 파일명 방지
        i = 1
        while dst.exists():
            stem, ext = os.path.splitext(fname)
            dst = OUT_DIR / f"{stem}_{i}{ext}"
            i += 1
        total = int(r.headers.get("Content-Length", 0))
        with open(dst, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=fname, leave=False) as bar:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    return dst

# -------- Selenium setup --------
opts = Options()
opts.add_argument("--headless=new")      # 필요 시 주석 처리해서 브라우저 보이게
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("--window-size=1400,900")

driver = webdriver.Chrome(options=opts)  # Selenium 4.6+면 경로 자동관리
wait = WebDriverWait(driver, 15)

try:
    # ===== 1) #model 밑의 모든 li 수집 (nth-child 제거) =====
    polite_get(driver, START_URL, wait, (By.CSS_SELECTOR, "#model"))
    # 직계 li 전부 → 그 하위 li 전부
    items = wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "#model > ul > li ul > li")
        )
    )
    if not items:
        raise RuntimeError("#model 하위에서 ul > li 항목을 찾지 못했습니다. 선택자/페이지를 확인하세요.")

    # ===== 2) 각 항목에서 a[href] 추출 (nth-child 사용 안 함) =====
    detail_links = []
    for li in items:
        try:
            a = li.find_element(By.CSS_SELECTOR, "a[href]")
            href = (a.get_attribute("href") or "").strip()
            if not href:
                continue
            abs_url = urljoin(BASE_URL, href)
            if abs_url not in detail_links:
                detail_links.append(abs_url)
        except:
            continue

    print(f"[INFO] 수집된 상세 링크: {len(detail_links)}개")

    # ===== 3) 상세 링크들에서 background-image 경로 추출 =====
    image_urls = []              # 절대 URL 리스트
    url_to_referer = {}          # 다운로드 시 Referer로 쓸 맵

    for link in tqdm(detail_links, desc="상세 페이지 파싱"):
        try:
            polite_get(driver, link, wait)  # 느슨히 로드 + 차단 감지
            # 기본 선택자
            time.sleep(2)
            divs = driver.find_elements(By.CSS_SELECTOR, "#carModelInfo > section.visual-wrap.design2023 > div")
            if not divs:
                # 폴백: 클래스 변형/구조 변경 대비
                divs = driver.find_elements(By.CSS_SELECTOR, "section.visual-wrap div[style*='background-image']")

            if not divs:
                print(f"[WARN] 배경이미지 div를 찾지 못함: {link}")
                continue

            found = False
            for div in divs:
                style = div.get_attribute("style") or ""
                rel = extract_bg_url(style)
                if not rel:
                    continue
                abs_img = urljoin(BASE_URL, rel)
                if abs_img not in url_to_referer:
                    image_urls.append(abs_img)
                    url_to_referer[abs_img] = link
                    found = True
            if not found:
                print(f"[WARN] style에서 url()을 추출하지 못함: {link}")

        except RuntimeError as e:
            # Cloudflare 1015 감지 시 종료
            print(str(e))
            break
        except Exception as e:
            print(f"[ERROR] {link}: {e}")

    image_urls = list(dict.fromkeys(image_urls))
    print(f"[INFO] 추출된 이미지 URL: {len(image_urls)}개")

    # ===== 4) 이미지 다운로드 (랜덤 지연 + Referer 포함) =====
    for img_url in tqdm(image_urls, desc="이미지 다운로드"):
        try:
            saved = download(img_url, referer=url_to_referer.get(img_url, BASE_URL))
            # print("saved:", saved)
        except Exception as e:
            print(f"[ERROR] download fail: {img_url} -> {e}")

    print(f"[DONE] 저장 폴더: {OUT_DIR.resolve()}")

finally:
    driver.quit()


[INFO] 수집된 상세 링크: 56개


상세 페이지 파싱:  12%|█▎        | 7/56 [01:00<06:50,  8.37s/it]

[WARN] 배경이미지 div를 찾지 못함: https://www.hyundai.com/kr/ko/e/vehicles/porter2-electric/intro


상세 페이지 파싱:  16%|█▌        | 9/56 [01:19<06:55,  8.84s/it]

[WARN] 배경이미지 div를 찾지 못함: https://www.hyundai.com/kr/ko/e/vehicles/avante-n/intro


...

[WARN] 배경이미지 div를 찾지 못함: https://www.hyundai.com/kr/ko/c/products/bus/universe-fuel-cell


상세 페이지 파싱: 100%|██████████| 56/56 [07:50<00:00,  8.41s/it]


[WARN] 배경이미지 div를 찾지 못함: https://www.hyundai.com/kr/ko/c/products/bus/universe-mobile-office
[INFO] 추출된 이미지 URL: 34개


이미지 다운로드: 100%|██████████| 34/34 [01:07<00:00,  1.97s/it]


[DONE] 저장 폴더: C:\Future\git\Project\Final_Project\image_data\insight&trends_images


In [ ]:
from pathlib import Path
from PIL import Image, ImageFile, UnidentifiedImageError, ImageOps
import numpy as np, io, cv2
from rembg import remove
from tqdm import tqdm

# ── HEIC/AVIF 열기(없으면 경고만) ──
try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
except Exception:
    print("[WARN] HEIC/AVIF를 열 수 없습니다. pillow-heif 설치 권장")

ImageFile.LOAD_TRUNCATED_IMAGES = True

SRC_DIR = Path("insight&trends_images")
OUT_DIR = Path("insight&trends_images_cropped")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ==== 기본(여유형, 잘림 최소화) 파라미터 ====
PADDING_RATIO = 0.12        # 최종 크롭 여유
ALPHA_THRESH   = 8          # 알파 임계(낮을수록 더 포용)
BASE_RATIO     = 0.03       # 형태 보정 커널 스케일
HGAP_RATIO     = 0.18       # 가로 연결 강화
DILATE_PX      = 24         # 경계 메꿈 세기(픽셀척도 → 내부 스케일로 변환)
MIN_AREA_RATIO = 0.0035     # 너무 작은 조각 제거
UNION_TOPK     = 4
ERODE_FOR_CROP_PX = 3       # ★ 크롭 전 연결 끊기 위한 선행 erode(px)
EXPAND_STEP_RATIO = 0.07    # 가장자리 닿음시 확장 step
EXPAND_MAX_ITER   = 3
SHAVE_ERODE_PX    = 10       # 가장자리 1~4px 정도 깎아 잔광 제거
FINAL_PAD_PX      = 8       # 마지막 타이트 크롭 여유
APPLY_FINAL_RECROP = True   # True로 바꾸면 마지막에 다시 타이트 크롭


VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp", ".heic", ".avif"}

# ---------------------- 유틸 ----------------------
def _odd(k: int) -> int: return int(k) | 1

def build_stable_mask(arr_rgba: np.ndarray, alpha_thresh=ALPHA_THRESH, base_ratio=BASE_RATIO, hgap_ratio=HGAP_RATIO):
    H, W = arr_rgba.shape[:2]
    alpha = arr_rgba[..., 3]
    mask = (alpha > alpha_thresh).astype(np.uint8) * 255

    k0 = max(_odd(int(max(H, W) * base_ratio)), 3)
    K0 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k0, k0))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, K0, iterations=1)
    mask = cv2.dilate(mask, K0, iterations=1)

    kw = max(_odd(int(W * hgap_ratio)), 5)
    kh = max(_odd(int(H * base_ratio * 0.6)), 3)
    Kh = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, kh))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, Kh, iterations=1)
    return mask

def pick_components(mask: np.ndarray, min_area_ratio=MIN_AREA_RATIO, strategy="center", union_topk=UNION_TOPK):
    H, W = mask.shape
    total = H * W
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    comps = []
    for c in cnts:
        area = cv2.contourArea(c)
        if area < total * min_area_ratio:
            continue
        M = cv2.moments(c)
        if M['m00'] > 0:
            cx = M['m10']/M['m00']; cy = M['m01']/M['m00']
        else:
            x, y, w, h = cv2.boundingRect(c)
            cx, cy = x + w/2, y + h/2
        dc = ((cx - W/2)/W)**2 + ((cy - H/2)/H)**2
        comps.append({"c": c, "area": area, "center_cost": dc})

    if not comps:
        return [], None, None

    if strategy == "largest":
        comps.sort(key=lambda d: d["area"], reverse=True)
    else:
        comps.sort(key=lambda d: (d["center_cost"], -d["area"]))

    best = comps[0]["c"]

    union_mask = np.zeros_like(mask)
    for i in range(min(len(comps), union_topk)):
        cv2.drawContours(union_mask, [comps[i]["c"]], 0, 255, -1)

    return comps, best, union_mask

def expand_if_touching(mask, x1, y1, x2, y2,
                       step_ratio=EXPAND_STEP_RATIO, max_iter=EXPAND_MAX_ITER,
                       edge_band=4, min_fg_ratio=0.012):
    H, W = mask.shape
    step = max(2, int(max(H, W) * step_ratio))
    for _ in range(max_iter):
        changed = False
        if x2 > x1:
            if (mask[y1:y2, x1:min(x2, x1+edge_band)] > 0).mean() > min_fg_ratio:
                x1 = max(0, x1 - step); changed = True
            if (mask[y1:y2, max(x1, x2-edge_band):x2] > 0).mean() > min_fg_ratio:
                x2 = min(W, x2 + step); changed = True
        if y2 > y1:
            if (mask[y1:min(y2, y1+edge_band), x1:x2] > 0).mean() > min_fg_ratio:
                y1 = max(0, y1 - step); changed = True
            if (mask[max(y1, y2-edge_band):y2, x1:x2] > 0).mean() > min_fg_ratio:
                y2 = min(H, y2 + step); changed = True
        if not changed: break
    return x1, y1, x2, y2

def _alpha_ring_stats(a: np.ndarray, ring_lo=5, ring_hi=160):
    ring = (a >= ring_lo) & (a <= ring_hi)
    if ring.sum() < 50:
        return None
    return float(a[ring].mean())

def dehalo_rgba(pil_rgba: Image.Image):
    """경계 밝기(링) 보고 흰/검 매트 자동 선택 후 straight-alpha 복원"""
    arr = np.asarray(pil_rgba).astype(np.float32)
    rgb = arr[..., :3]
    a   = arr[..., 3:4] / 255.0
    ring_mean = _alpha_ring_stats(arr[..., 3])
    if ring_mean is None:
        matte = "black"
    else:
        # 경계가 밝으면 'white' 쪽이 대체로 자연스러움
        matte = "white" if ring_mean > 130 else "black"

    eps = 1e-6
    if matte == "black":
        rgb = np.where(a > eps, np.clip(rgb / (a + eps), 0, 255), 0)
    else:
        rgb = np.where(a > eps, np.clip((rgb - (1 - a) * 255) / (a + eps), 0, 255), 255)
    arr[..., :3] = rgb
    return Image.fromarray(arr.astype(np.uint8)).convert("RGBA")

def shave_alpha(pil_rgba: Image.Image, erode_px=SHAVE_ERODE_PX, blur_px=0):
    arr = np.asarray(pil_rgba).copy()
    a = arr[..., 3]
    if erode_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_px*2+1, erode_px*2+1))
        a = cv2.erode(a, k, iterations=1)
    if blur_px > 0:
        a = cv2.GaussianBlur(a, (0, 0), blur_px)
    arr[..., 3] = a
    return Image.fromarray(arr).convert("RGBA")

def final_alpha_recrop(pil_rgba: Image.Image, pad_px=FINAL_PAD_PX, thr=ALPHA_THRESH):
    arr = np.asarray(pil_rgba)
    a = arr[..., 3]
    ys, xs = np.where(a > thr)
    if len(xs) == 0:
        return pil_rgba
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    x1 = max(0, x1 - pad_px); y1 = max(0, y1 - pad_px)
    x2 = min(arr.shape[1], x2 + pad_px + 1); y2 = min(arr.shape[0], y2 + pad_px + 1)
    return Image.fromarray(arr[y1:y2, x1:x2, :]).convert("RGBA")

def solidify_edge_colors(pil_rgba: Image.Image, opaque_thr: int = 230, inpaint_radius: int = 8):
    """
    반투명 가장자리(0<alpha<opaque_thr)의 RGB를 내부 불투명 영역 색으로 '채움'.
    배경색 오염(검은/흰 후광) 제거용.
    """
    arr = np.asarray(pil_rgba).copy()
    rgb = arr[..., :3]
    a   = arr[..., 3]

    # 반투명 링만 인페인트 대상으로 선택
    ring_mask = ((a > 0) & (a < opaque_thr)).astype(np.uint8) * 255  # 0~254 → 대상
    # OpenCV는 3채널만 인페인트하므로 RGB만 전달
    filled = cv2.inpaint(rgb, ring_mask, inpaintRadius=inpaint_radius, flags=cv2.INPAINT_TELEA)
    # 링 영역에만 채운 색 적용
    rgb = np.where(ring_mask[..., None] > 0, filled, rgb)

    arr[..., :3] = rgb
    return Image.fromarray(arr).convert("RGBA")

# ---------------------- 크롭 본체(각도 보정 없음) ----------------------
def crop_objects_advanced(
    img_rgba: Image.Image,
    pad_ratio: float = 0.12,
    alpha_thresh: int = 8,
    base_ratio: float = 0.02,
    hgap_ratio: float = 0.10,
    min_area_ratio: float = 0.004,
    multi_mode: str = "largest",
    # ↓ 새로 추가: 최소 픽셀 여유 + 확장 강도
    min_pad_px: int = 48,
    expand_step_ratio: float = 0.10,
    expand_iters: int = 5,
    expand_edge_band: int = 10,
    expand_min_fg_ratio: float = 0.003,
    # 과거 호환
    dilate_px: int | None = 18,
    union_topk: int = 3,
    deskew: bool = False, max_rot_deg: float = 0.0,
    ar_threshold: float = 0.0, min_ar_gain: float = 0.0, rotate_extra_ratio: float = 0.0,
):
    img_rgba = ImageOps.exif_transpose(img_rgba)
    if img_rgba.mode != "RGBA":
        img_rgba = img_rgba.convert("RGBA")

    arr = np.asarray(img_rgba); H, W = arr.shape[:2]

    if dilate_px is not None:
        px_ratio = max(dilate_px / max(H, W), 1.0 / max(H, W))
        base_ratio = max(base_ratio, px_ratio * 1.2)
        hgap_ratio = max(hgap_ratio, px_ratio * 4.0)

    mask = build_stable_mask(arr, alpha_thresh=alpha_thresh,
                             base_ratio=base_ratio, hgap_ratio=hgap_ratio)

    comps, best, union_mask = pick_components(mask, min_area_ratio=min_area_ratio,
                                              strategy="center", union_topk=union_topk)

    def _safe_expand(m, x1, y1, x2, y2):
        return expand_if_touching(
            m, x1, y1, x2, y2,
            step_ratio=expand_step_ratio,
            max_iter=expand_iters,
            edge_band=expand_edge_band,
            min_fg_ratio=expand_min_fg_ratio,
        )

    if not comps:
        alpha = arr[..., 3]
        ys, xs = np.where(alpha > alpha_thresh)
        if len(xs) == 0: return []
        x1, x2 = xs.min(), xs.max(); y1, y2 = ys.min(), ys.max()
        bw, bh = (x2 - x1 + 1), (y2 - y1 + 1)
        px = max(int(bw * pad_ratio), min_pad_px)
        py = max(int(bh * pad_ratio), min_pad_px)
        x1 = max(0, x1 - px); y1 = max(0, y1 - py)
        x2 = min(W, x2 + px + 1); y2 = min(H, y2 + py + 1)
        comp_mask = (alpha > alpha_thresh).astype(np.uint8) * 255
        x1, y1, x2, y2 = _safe_expand(comp_mask, x1, y1, x2, y2)
        return [Image.fromarray(arr[y1:y2, x1:x2, :], "RGBA")]

    targets = comps if multi_mode == "separate" else [{"c": best}]
    results = []

    for t in targets:
        c = t["c"]
        x, y, w, h = cv2.boundingRect(c)
        pad = max(int(max(w, h) * pad_ratio), min_pad_px)
        x1 = max(0, x - pad); y1 = max(0, y - pad)
        x2 = min(W, x + w + pad); y2 = min(H, y + h + pad)

        # 얇거나 작은 경우 union bbox로 교체 + 같은 패딩 규칙 적용
        if union_mask is not None:
            ys_u, xs_u = np.where(union_mask > 0)
            if len(xs_u) > 0:
                ux1, ux2 = xs_u.min(), xs_u.max()
                uy1, uy2 = ys_u.min(), ys_u.max()
                skinny = max(w, h) / max(1, min(w, h)) > 8.0
                small  = (w*h) / (H*W) < 0.015
                if skinny or small:
                    pw = max(int((ux2 - ux1 + 1) * pad_ratio), min_pad_px)
                    ph = max(int((uy2 - uy1 + 1) * pad_ratio), min_pad_px)
                    x1 = max(0, ux1 - pw); y1 = max(0, uy1 - ph)
                    x2 = min(W, ux2 + pw + 1); y2 = min(H, uy2 + ph + 1)

        comp_mask = np.zeros((H, W), np.uint8)
        cv2.drawContours(comp_mask, [c], 0, 255, -1)

        x1, y1, x2, y2 = _safe_expand(comp_mask, x1, y1, x2, y2)
        crop = arr[y1:y2, x1:x2, :]
        if crop.size > 0: results.append(Image.fromarray(crop, "RGBA"))

    return results

# ---------------------- 저장 ----------------------
def _sanitize_stem(stem: str) -> str:
    stem = stem.strip()
    for ch in '<>:"/\\|?*':
        stem = stem.replace(ch, '')
    return ' '.join(stem.split())

def save_crops(save_stem: str, crops: list[Image.Image], out_dir: Path):
    """save_stem 이름으로 PNG 저장. 이미 있으면 스킵."""
    save_stem = _sanitize_stem(save_stem)
    saved_any = False
    if len(crops) == 1:
        out_path = out_dir / f"{save_stem}.png"
        if out_path.exists():
            print(f"[SKIP] {out_path.name} 이미 존재")
        else:
            crops[0].save(out_path, format="PNG", optimize=True)
            saved_any = True
    else:
        for i, im in enumerate(crops, 1):
            out_path = out_dir / f"{save_stem}_{i:02d}.png"
            if out_path.exists():
                print(f"[SKIP] {out_path.name} 이미 존재"); continue
            im.save(out_path, format="PNG", optimize=True)
            saved_any = True
    return saved_any

# ---------------------- 선택 실행(확장자 없이 지정) ----------------------
TARGET_STEMS = [
    "뉴파워트럭", "더 뉴 파비스", "더 뉴 엑시언트", "아이오닉 5 N",
    "유니버스 모바일 오피스", "일렉시티 이층버스", "포터 II Electric", "포터 II",
]

# stem(소문자) → 경로들
index = {}
for p in SRC_DIR.iterdir():
    if p.is_file() and p.suffix.lower() in VALID_EXTS:
        index.setdefault(p.stem.lower(), []).append(p)

EXT_PRIORITY = {".heic":0, ".avif":1, ".png":2, ".jpg":3, ".jpeg":4, ".webp":5, ".gif":6, ".bmp":7}
pick_best = lambda paths: sorted(paths, key=lambda x: (EXT_PRIORITY.get(x.suffix.lower(), 99), -x.stat().st_mtime))[0]

# 요청 스템과 실제 경로를 매칭해 “(저장용 이름, 파일 경로)” 리스트 생성
worklist: list[tuple[str, Path]] = []
for req in TARGET_STEMS:
    paths = index.get(req.lower())
    if not paths:
        print(f"[MISS] '{req}' 에 해당하는 파일을 못 찾았습니다."); continue
    worklist.append((req, pick_best(paths)))

print(f"[INFO] 선택된 대상: {len(worklist)}개 / 요청 {len(TARGET_STEMS)}개")

# ---------------------- 메인 루프 ----------------------
for save_name, img_path in tqdm(worklist, desc="rembg crop"):
    try:
        with Image.open(img_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGBA")
            out = remove(im)
            cut = (Image.open(io.BytesIO(out)).convert("RGBA")
                   if isinstance(out, (bytes, bytearray))
                   else (out.convert("RGBA") if isinstance(out, Image.Image)
                         else Image.fromarray(
                              (np.dstack([out, np.full(out.shape[:2], 255, out.dtype)])
                               if (out.ndim==3 and out.shape[2]==3) else
                               (np.stack([out,out,out,np.full_like(out,255)],-1))),
                              mode="RGBA")))

            crops = crop_objects_advanced(
                cut,
                pad_ratio=PADDING_RATIO,
                alpha_thresh=12,
                base_ratio=0.02,
                hgap_ratio=0.10,
                min_area_ratio=0.005,
                multi_mode="largest",
                deskew=True,
                dilate_px=20,
            )
            if not crops:
                print(f"[WARN] {img_path.name}: 유효 물체 없음"); continue

            outs = []
            for c in crops:
                c = dehalo_rgba(c)
                c = shave_alpha(c, erode_px=2)
                # final_alpha_recrop는 쓰지 않거나 pad_px 크게(여백 유지하려면 0~2 권장)
                # c = final_alpha_recrop(c, pad_px=0)
                outs.append(c)

            _ = save_crops(save_name, outs, OUT_DIR)  # ← ★ 저장용 이름 고정!

    except UnidentifiedImageError as e:
        print(f"[ERROR] {img_path.name}: 이미지 식별 실패 - {e}")
    except Exception as e:
        print(f"[ERROR] {img_path.name}: {e}")

print(f"[DONE] 저장 폴더: {OUT_DIR.resolve()}")


[INFO] 선택된 대상: 8개 / 요청 8개


rembg crop:   0%|          | 0/8 [00:00<?, ?it/s]C:\Users\jinhy\AppData\Local\Temp\ipykernel_9124\3412922711.py:270: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  if crop.size > 0: results.append(Image.fromarray(crop, "RGBA"))
rembg crop:  12%|█▎        | 1/8 [00:02<00:14,  2.13s/it]

[SKIP] 뉴파워트럭.png 이미 존재


rembg crop:  38%|███▊      | 3/8 [00:06<00:10,  2.16s/it]

[SKIP] 더 뉴 엑시언트.png 이미 존재


...

[SKIP] 유니버스 모바일 오피스.png 이미 존재


rembg crop:  75%|███████▌  | 6/8 [00:13<00:04,  2.15s/it]

[SKIP] 일렉시티 이층버스.png 이미 존재


rembg crop: 100%|██████████| 8/8 [00:18<00:00,  2.36s/it]

[DONE] 저장 폴더: C:\Future\git\Project\Final_Project\image_data\insight&trends_images_cropped
